# Solutions — Reading and Writing Data (Hard 15)

In [ ]:
CATALOG = "workspace"   # Free Edition default. Paid workspace? Change to your catalog (e.g. "main")
SCHEMA  = "default"

from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_trips        = spark.table("samples.nyctaxi.trips")

# Create a Unity Catalog Volume for intermediate files
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.practice_io")

# Base path for all intermediate files in this notebook
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/practice_io/spark_io_practice"

## Solution 1 — Write CSV and Read it Back with inferSchema

In [ ]:
csv_path = f"{BASE}/transactions_csv"
df_transactions.write.option("header", "true").mode("overwrite").csv(csv_path)
result_1 = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_path)


## Solution 2 — Read CSV with Explicit Schema (no inferSchema)

In [ ]:
csv_path_2 = f"{BASE}/transactions_csv_p2"
df_transactions.limit(500).write.option("header", "true").mode("overwrite").csv(csv_path_2)

explicit_schema = T.StructType([
    T.StructField("transactionID", T.IntegerType(), True),
    T.StructField("dateTime",      T.StringType(),  True),
    T.StructField("product",       T.StringType(),  True),
    T.StructField("quantity",      T.IntegerType(), True),
    T.StructField("unitPrice",     T.DoubleType(),  True),
    T.StructField("totalPrice",    T.DoubleType(),  True),
    T.StructField("paymentMethod", T.StringType(),  True),
    T.StructField("cardNumber",    T.StringType(),  True),
    T.StructField("customerID",    T.IntegerType(), True),
    T.StructField("franchiseID",   T.IntegerType(), True),
])

result_2 = (
    spark.read.option("header", "true")
    .schema(explicit_schema)
    .csv(csv_path_2)
)


## Solution 3 — Write and Read JSON

In [ ]:
json_path = f"{BASE}/transactions_json"
df_transactions.write.mode("overwrite").json(json_path)
result_3 = spark.read.json(json_path).select("transactionID")


## Solution 4 — Write and Read Parquet

In [ ]:
parquet_path = f"{BASE}/transactions_parquet"
df_transactions.write.mode("overwrite").parquet(parquet_path)
result_4 = spark.read.parquet(parquet_path).select("transactionID")


## Solution 5 — Partitioned Write

In [ ]:
partitioned_path = f"{BASE}/transactions_partitioned"
df_transactions.write.mode("overwrite").partitionBy("franchiseID").parquet(partitioned_path)
result_5 = spark.read.parquet(partitioned_path).select("transactionID", "franchiseID")


## Solution 6 — Custom Separator CSV (pipe-delimited)

In [ ]:
import pyspark.sql.functions as F

pipe_path = f"{BASE}/pipe_csv"
sample_data = spark.createDataFrame(
    [(1, "Alpha", 99.9), (2, "Beta", 49.5), (3, "Gamma", 19.0)],
    ["id", "name", "value"]
)
sample_data.write.option("sep", "|").option("header", "true").mode("overwrite").csv(pipe_path)
result_6 = spark.read.option("sep", "|").option("header", "true").csv(pipe_path)


## Solution 7 — saveAsTable (Managed Table)

In [ ]:
TABLE_NAME = f"{CATALOG}.{SCHEMA}.transactions_practice_table"
df_transactions.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)
result_7 = spark.table(TABLE_NAME).select("transactionID")

## Solution 8 — Schema Evolution with mergeSchema

In [ ]:
parquet_path = f"{BASE}/transactions_parquet"
with_extra = df_transactions.withColumn("is_flagged", F.lit(False))
with_extra.write.mode("append").option("mergeSchema", "true").parquet(parquet_path)
result_8 = spark.read.option("mergeSchema", "true").parquet(parquet_path).select("transactionID", "product")
